# 🔬 TCGA-LUAD: WSI Analysis & Feature Extraction

**Deskripsi:**
Notebook ini mencakup:
1. **Setup** — Mount Google Drive, install dependencies
2. **Visualisasi WSI** — Info WSI dari manifest CSV (jumlah, ukuran, tipe, magnifikasi)
3. **Analisis Patches_Macenko** — Statistik dataset, distribusi patch, contoh visual
4. **Ekstraksi Fitur Swin-Tiny** — Per WSI → simpan ke `Fitur_Swin_Tiny/*.h5`
5. **Ekstraksi Fitur DenseNet201** — Per WSI → simpan ke `Fitur_DenseNet201/*.h5`

> ⚠️ Notebook ini dirancang untuk **Google Colab** dengan Google Drive mounted.
> Mendukung **resume otomatis** jika runtime terputus.

**Struktur Folder:**
```
MyDrive/TCGA-LUAD-WSI/
├── Patches_Macenko/
│   ├── train/
│   │   ├── Mutant/    └── {WSI_ID}.zip  ← ZIP berisi patch PNG per WSI
│   │   └── Wildtype/  └── {WSI_ID}.zip
│   ├── test/          (idem)
│   ├── macenko_manifest.csv
│   └── macenko_report_all.csv
├── Fitur_Swin_Tiny/     ← OUTPUT: {WSI_ID}.h5
└── Fitur_DenseNet201/   ← OUTPUT: {WSI_ID}.h5
```

---
## 📦 Sel 0: Install Dependencies & Mount Google Drive

In [ ]:
# ============================================================
# INSTALL DEPENDENCIES
# ============================================================
!pip install timm h5py tqdm Pillow -q
print("✅ Dependencies installed")

In [ ]:
# ============================================================
# MOUNT GOOGLE DRIVE (hanya jika berjalan di Colab)
# Dengan retry otomatis jika autentikasi gagal (MessageError)
# ============================================================
import os
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("ℹ️  Bukan Colab — lewati mount Drive (atur ROOT_MANUAL di sel berikutnya)")

if IN_COLAB:
    def _mounted():
        return os.path.isdir('/content/drive/MyDrive')
    if _mounted():
        print("✅ Drive sudah ter-mount")
    else:
        for attempt, force in enumerate([False, True], 1):
            try:
                drive.mount('/content/drive', force_remount=force)
                break
            except Exception as e:
                print(f"⚠️ Percobaan {attempt} gagal: {type(e).__name__}: {e}")
        if _mounted():
            print("✅ Google Drive mounted at /content/drive")
        else:
            print("""
❌ Mount gagal. Coba berurutan:
  1. Saat popup 'Permit this notebook to access your Google Drive' muncul,
     klik Connect → pilih AKUN YANG SAMA dengan akun Colab → izinkan semua.
  2. Izinkan popup & cookie pihak ketiga untuk colab.research.google.com
     (atau coba buka di Chrome tanpa mode incognito / tanpa ad-blocker).
  3. Menu Runtime → Disconnect and delete runtime, lalu jalankan ulang sel ini.
  4. Alternatif tanpa popup: klik ikon folder di sidebar kiri → ikon 'Mount Drive'.
""")


In [ ]:
# ============================================================
# KONFIGURASI PATH
# ROOT dicari otomatis. Jika gagal, isi ROOT_MANUAL.
# ============================================================
import os

ROOT_MANUAL = None   # contoh: "/content/drive/MyDrive/TCGA-LUAD-WSI"

def find_root():
    if ROOT_MANUAL:
        return ROOT_MANUAL
    base = "/content/drive/MyDrive"
    cands = [f"{base}/TCGA-LUAD-WSI", f"{base}/TCGA-LUAD"]
    for c in cands:
        if os.path.isdir(f"{c}/Patches_Macenko"):
            return c
    # cari folder 'Patches_Macenko' sampai kedalaman 3
    if os.path.isdir(base):
        for dp, dn, _ in os.walk(base):
            depth = dp[len(base):].count(os.sep)
            if "Patches_Macenko" in dn:
                return dp
            if depth >= 3:
                dn[:] = []
    return cands[0]   # fallback (akan dilaporkan tidak ada)

ROOT          = find_root()
PATCH_DIR     = f"{ROOT}/Patches_Macenko"

# Output folder fitur
FEAT_SWIN     = f"{ROOT}/Fitur_Swin_Tiny"
FEAT_DENSE    = f"{ROOT}/Fitur_DenseNet201"

# Splits & kelas
SPLITS        = ["train", "validation", "test"]
CLASSES       = ["Mutant", "Wildtype"]

# Hyperparameter ekstraksi
PATCH_SIZE    = 224
BATCH_SIZE    = 64    # kurangi ke 32 jika OOM
NUM_WORKERS   = 2

print(f"ROOT          : {ROOT}")
print(f"PATCH_DIR     : {PATCH_DIR}")
print(f"ROOT exists   : {os.path.exists(ROOT)}")
print(f"PATCH_DIR ok  : {os.path.exists(PATCH_DIR)}")

if not os.path.isdir(PATCH_DIR):
    print("\n❌ Folder Patches_Macenko TIDAK ditemukan.")
    print("   Isi folder induk:", os.listdir(os.path.dirname(ROOT)) if os.path.isdir(os.path.dirname(ROOT)) else "(tidak ada)")
    print("   👉 Jika folder dibagikan orang lain: buka link Drive → klik nama folder →")
    print("      'Organize' → 'Add shortcut' ke My Drive, lalu jalankan ulang sel ini.")
    raise FileNotFoundError(PATCH_DIR)

# Buat output folder (hanya setelah PATCH_DIR valid)
os.makedirs(FEAT_SWIN,  exist_ok=True)
os.makedirs(FEAT_DENSE, exist_ok=True)
print("\n📁 Konfigurasi selesai.")


---
## 🔎 Sel 1: Visualisasi WSI dari Manifest CSV

In [ ]:
# ============================================================
# IMPORT LIBRARY
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import random
from pathlib import Path
from PIL import Image
from collections import defaultdict
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# Palette warna
PALETTE = {
    'Mutant'     : '#E74C3C',
    'Wildtype'   : '#3498DB',
    'train'      : '#2ECC71',
    'validation' : '#F39C12',
    'test'       : '#9B59B6',
}
ACCENT = '#1ABC9C'
BG     = '#0D1117'
PANEL  = '#161B22'

def style_ax(ax, title):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors='#8B949E', labelsize=9)
    ax.spines[:].set_color('#30363D')
    ax.set_title(title, color='white', fontsize=11, fontweight='bold', pad=8)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_color('#8B949E')

print("✅ Library dan style siap")

In [ ]:
# ============================================================
# LOAD MANIFEST CSV (opsional — hanya untuk informasi)
# ============================================================
manifest_df, manifest_path = None, None
for p in [f"{PATCH_DIR}/macenko_manifest.csv", f"{PATCH_DIR}/macenko_report_all.csv"]:
    if os.path.exists(p):
        manifest_df = pd.read_csv(p); manifest_path = p
        print(f"✅ Loaded: {p}  | Shape: {manifest_df.shape}")
        print("📋 Kolom:", manifest_df.columns.tolist())
        display(manifest_df.head())
        break
else:
    print("ℹ️  Manifest CSV tidak ditemukan — tidak masalah, wsi_df dibuat dari scan folder.")


In [ ]:
# ============================================================
# SCAN PATCHES_MACENKO → BUAT wsi_df
# Mendukung 2 format:
#   (a) ZIP per WSI : {split}/{class}/{WSI_ID}.zip   ← format dataset Anda
#   (b) Folder      : {split}/{class}/{WSI_ID}/*.png
# Folder lain (qc, manifests, reference, dll.) diabaikan.
# ============================================================
import zipfile
from tqdm.auto import tqdm

SPLIT_ALIASES = {'train':'train','training':'train',
                 'val':'validation','valid':'validation','validation':'validation',
                 'test':'test','testing':'test'}
CLASS_ALIASES = {'mutant':'Mutant','mut':'Mutant','egfr_mutant':'Mutant','egfr-mutant':'Mutant',
                 'wildtype':'Wildtype','wild_type':'Wildtype','wild-type':'Wildtype',
                 'wt':'Wildtype','egfr_wildtype':'Wildtype','egfr-wildtype':'Wildtype'}
IMG_EXT = {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}

def _norm(s): return s.strip().lower().replace(' ', '_')

def is_img_name(n):
    base = os.path.basename(n)
    return (os.path.splitext(base)[1].lower() in IMG_EXT
            and not base.startswith('.') and '__MACOSX' not in n)

def list_zip_images(zip_path):
    with zipfile.ZipFile(zip_path) as zf:
        return [n for n in zf.namelist() if is_img_name(n)]

def read_img_from_zip(zip_path, name):
    with zipfile.ZipFile(zip_path) as zf:
        with zf.open(name) as f:
            return Image.open(io.BytesIO(f.read())).convert('RGB')

import io
entries, skipped, ignored_dirs = [], [], []
for sp_name in sorted(os.listdir(PATCH_DIR)):
    sp_dir = os.path.join(PATCH_DIR, sp_name)
    split = SPLIT_ALIASES.get(_norm(sp_name))
    if not os.path.isdir(sp_dir):
        continue
    if split is None:
        ignored_dirs.append(sp_name); continue
    for c_name in sorted(os.listdir(sp_dir)):
        c_dir = os.path.join(sp_dir, c_name)
        cls = CLASS_ALIASES.get(_norm(c_name))
        if cls is None or not os.path.isdir(c_dir):
            continue
        for e in sorted(os.listdir(c_dir)):
            ep = os.path.join(c_dir, e)
            if e.lower().endswith('.zip'):
                entries.append((split, cls, e[:-4], ep, 'zip'))
            elif os.path.isdir(ep):
                entries.append((split, cls, e, ep, 'dir'))

print(f"🔎 Ditemukan {len(entries)} kandidat WSI (zip/folder). Membaca daftar patch...")
if ignored_dirs:
    print(f"   (folder non-split diabaikan: {ignored_dirs})")

wsi_info = []
for split, cls, wsi_id, src, kind in tqdm(entries, desc="Scan"):
    try:
        if kind == 'zip':
            names = list_zip_images(src)
            sample = names[0] if names else None
        else:
            names = [str(q) for q in Path(src).rglob('*') if q.suffix.lower() in IMG_EXT]
            sample = names[0] if names else None
        if not names:
            skipped.append((wsi_id, 'tidak ada gambar')); continue
        if kind == 'zip':
            im = read_img_from_zip(src, sample)
        else:
            im = Image.open(sample)
        w, h, mode = im.width, im.height, im.mode
    except Exception as ex:
        skipped.append((wsi_id, f'{type(ex).__name__}: {ex}')); continue
    wsi_info.append({
        'wsi_id': wsi_id, 'patient_id': wsi_id[:12], 'split': split, 'label': cls,
        'n_patches': len(names), 'patch_w': w, 'patch_h': h, 'img_mode': mode,
        'wsi_dir': src,            # path ZIP atau folder (nama kolom dipertahankan)
        'src_type': kind, 'sample_patch': sample,
    })

if not wsi_info:
    print("❌ Tidak ada WSI valid. Isi PATCH_DIR:", sorted(os.listdir(PATCH_DIR)))
    print("   Struktur diharapkan: Patches_Macenko/{train|test}/{Mutant|Wildtype}/{WSI_ID}.zip")
    raise RuntimeError("wsi_df kosong — cek struktur folder / PATCH_DIR di atas.")

wsi_df = pd.DataFrame(wsi_info)
print(f"\n📊 Total WSI  : {len(wsi_df)}")
print(f"   Total patch: {wsi_df['n_patches'].sum():,}")
print(wsi_df.groupby(['split', 'label']).agg(WSI=('wsi_id', 'count'), Patch=('n_patches', 'sum')))
if skipped:
    print(f"\n⚠️  {len(skipped)} entri dilewati, contoh: {skipped[:3]}")

# Cek kebocoran data: pasien yang sama berada di >1 split
leak = wsi_df.groupby('patient_id')['split'].nunique()
leak = leak[leak > 1]
if len(leak):
    print(f"\n⚠️  {len(leak)} pasien punya slide di >1 split (kebocoran data train/test):")
    display(wsi_df[wsi_df['patient_id'].isin(leak.index)][['patient_id', 'wsi_id', 'split', 'label']])
else:
    print("\n✅ Tidak ada kebocoran pasien antar split.")
dup = wsi_df[wsi_df['wsi_id'].duplicated(keep=False)]
if len(dup):
    print("⚠️  WSI_ID dobel:"); display(dup[['wsi_id', 'split', 'label']])
display(wsi_df.head())


In [ ]:
# ============================================================
# DASHBOARD VISUALISASI WSI
# ============================================================
fig = plt.figure(figsize=(22, 18), facecolor=BG)
fig.suptitle('🔬 TCGA-LUAD: WSI Dataset Overview',
             fontsize=20, fontweight='bold', color='white', y=0.99)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.38)

# ── 1. WSI per Split ──────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
split_c = wsi_df.groupby('split')['wsi_id'].count().reindex(['train','validation','test'], fill_value=0)
bars1 = ax1.bar(split_c.index, split_c.values,
                color=[PALETTE.get(s, ACCENT) for s in split_c.index],
                edgecolor='#30363D', linewidth=0.7)
for bar, v in zip(bars1, split_c.values):
    ax1.text(bar.get_x() + bar.get_width()/2, v + 0.2,
             str(v), ha='center', va='bottom', color='white', fontweight='bold', fontsize=12)
style_ax(ax1, '📁 Jumlah WSI per Split')
ax1.set_ylabel('Jumlah WSI', color='#8B949E')

# ── 2. Distribusi Label (Pie) ─────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
label_c = wsi_df.groupby('label')['wsi_id'].count()
wedges, texts, autotexts = ax2.pie(
    label_c.values, labels=label_c.index,
    colors=[PALETTE.get(l, ACCENT) for l in label_c.index],
    autopct='%1.1f%%', startangle=90,
    textprops={'color': 'white', 'fontsize': 11},
    wedgeprops={'edgecolor': BG, 'linewidth': 2},
    pctdistance=0.75
)
for at in autotexts:
    at.set_fontweight('bold')
ax2.set_facecolor(PANEL)
ax2.set_title('🏷️ Distribusi Label WSI', color='white', fontsize=11, fontweight='bold', pad=8)

# ── 3. Boxplot Patch per WSI per Kelas ───────────────────
ax3 = fig.add_subplot(gs[0, 2])
data_box = [wsi_df[wsi_df['label'] == cls]['n_patches'].values for cls in CLASSES]
bp = ax3.boxplot(data_box, patch_artist=True,
                 medianprops={'color': 'white', 'linewidth': 2},
                 whiskerprops={'color': '#8B949E'},
                 capprops={'color': '#8B949E'},
                 flierprops={'marker': 'o', 'color': '#8B949E', 'alpha': 0.5})
ax3.set_xticklabels(CLASSES)
for box, cls in zip(bp['boxes'], CLASSES):
    box.set_facecolor(PALETTE[cls]); box.set_alpha(0.7)
style_ax(ax3, '📊 Distribusi Patch per WSI')
ax3.set_ylabel('Jumlah Patch', color='#8B949E')

# ── 4. Grouped bar: Patch per Split × Label ───────────────
ax4 = fig.add_subplot(gs[1, 0:2])
pivot = wsi_df.groupby(['split','label'])['n_patches'].sum().unstack(fill_value=0)
pivot = pivot.reindex(['train','validation','test'], fill_value=0)
x_pos = np.arange(len(pivot)); width = 0.35
for i, col in enumerate(pivot.columns):
    offset = (i - len(pivot.columns)/2 + 0.5) * width
    bars = ax4.bar(x_pos + offset, pivot[col], width, label=col,
                   color=PALETTE.get(col, ACCENT), edgecolor='#30363D',
                   linewidth=0.5, alpha=0.85)
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax4.text(bar.get_x() + bar.get_width()/2, h + 50,
                     f'{int(h):,}', ha='center', va='bottom',
                     color='white', fontsize=8, fontweight='bold')
ax4.set_xticks(x_pos); ax4.set_xticklabels(pivot.index)
ax4.legend(facecolor='#21262D', edgecolor='#30363D', labelcolor='white', fontsize=9)
style_ax(ax4, '🔢 Total Patch per Split × Label')
ax4.set_ylabel('Total Patch', color='#8B949E')

# ── 5. Ukuran Patch Distribution ─────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
size_counts = wsi_df.apply(lambda r: f"{int(r['patch_w'])}×{int(r['patch_h'])}", axis=1).value_counts()
ax5.barh(size_counts.index, size_counts.values, color=ACCENT,
         edgecolor='#30363D', linewidth=0.5)
for i, v in enumerate(size_counts.values):
    ax5.text(v + 0.1, i, f' {v} WSI', va='center', color='white', fontsize=9)
style_ax(ax5, '📐 Ukuran Patch (px)')
ax5.set_xlabel('Jumlah WSI', color='#8B949E')

# ── 6. Tabel Statistik Ringkas ────────────────────────────
ax6 = fig.add_subplot(gs[2, :])
ax6.set_facecolor(PANEL); ax6.axis('off')
ax6.set_title('📋 Statistik Ringkas Dataset', color='white',
              fontsize=11, fontweight='bold', pad=8)

stats = wsi_df.groupby('label').agg(
    Jml_WSI      = ('wsi_id',    'count'),
    Total_Patch  = ('n_patches', 'sum'),
    Min_Patch    = ('n_patches', 'min'),
    Max_Patch    = ('n_patches', 'max'),
    Mean_Patch   = ('n_patches', 'mean'),
    Median_Patch = ('n_patches', 'median'),
).reset_index()
stats['Mean_Patch']   = stats['Mean_Patch'].round(1)
stats['Median_Patch'] = stats['Median_Patch'].round(1)
total_row = pd.DataFrame([{
    'label': 'TOTAL', 'Jml_WSI': len(wsi_df),
    'Total_Patch': wsi_df['n_patches'].sum(),
    'Min_Patch': wsi_df['n_patches'].min(),
    'Max_Patch': wsi_df['n_patches'].max(),
    'Mean_Patch': round(wsi_df['n_patches'].mean(), 1),
    'Median_Patch': wsi_df['n_patches'].median(),
}])
stats = pd.concat([stats, total_row], ignore_index=True)

col_labels = ['Label','Jml WSI','Total Patch','Min Patch','Max Patch','Mean','Median']
tbl = ax6.table(cellText=stats.values.tolist(), colLabels=col_labels,
                cellLoc='center', loc='center', bbox=[0.02, 0.0, 0.96, 1.0])
tbl.auto_set_font_size(False); tbl.set_fontsize(10)
for (row, col), cell in tbl.get_celld().items():
    cell.set_facecolor('#21262D' if row % 2 == 0 else PANEL)
    cell.set_edgecolor('#30363D')
    cell.set_text_props(color='white')
    if row == 0:
        cell.set_facecolor('#1F6FEB')
        cell.set_text_props(color='white', fontweight='bold')
    if row == len(stats):
        cell.set_facecolor('#238636')
        cell.set_text_props(color='white', fontweight='bold')

plt.savefig('/tmp/wsi_overview.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("\n📊 WSI Overview dashboard disimpan.")

In [ ]:
# ============================================================
# TABEL DETAIL WSI — Scrollable
# ============================================================
disp = wsi_df[['wsi_id','split','label','n_patches','patch_w','patch_h','img_mode']].copy()
disp.columns = ['WSI ID','Split','Label','# Patch','W (px)','H (px)','Mode']
disp = disp.sort_values(['Split','Label','WSI ID']).reset_index(drop=True)

def color_label(v):
    c = PALETTE.get(v, '#888')
    return f'background-color: {c}25; color: {c}; font-weight:bold'
def color_split(v):
    c = PALETTE.get(v, '#888')
    return f'color:{c}; font-weight:bold'

_st = disp.style
_map = _st.map if hasattr(_st, 'map') else _st.applymap   # pandas >=2.1 pakai .map
_map(color_label, subset=['Label'])
_map(color_split, subset=['Split'])
styled = (_st
    .format({'# Patch': '{:,}'})
    .set_table_styles([{'selector':'thead th',
        'props':[('background','#1F6FEB'),('color','white'),('font-weight','bold')]}])
    .set_caption(f"Detail {len(disp)} WSI dari Patches_Macenko")
)
display(HTML(f"<div style='max-height:450px;overflow-y:auto'>{styled.to_html()}</div>"))

---
## 🖼️ Sel 2: Analisis & Visualisasi Dataset Patches_Macenko

In [ ]:
# ============================================================
# STATISTIK KESELURUHAN
# ============================================================
print("═" * 65)
print("    📊 STATISTIK KESELURUHAN DATASET PATCHES_MACENKO")
print("═" * 65)
print(f"  Total WSI          : {len(wsi_df)}")
print(f"  Total Patch        : {wsi_df['n_patches'].sum():,}")
print(f"  Splits             : {', '.join(wsi_df['split'].unique())}")
print(f"  Kelas              : {', '.join(wsi_df['label'].unique())}")
print()
for sp in SPLITS:
    sdf = wsi_df[wsi_df['split'] == sp]
    if len(sdf) == 0: continue
    print(f"  [{sp.upper():<12}]  WSI={len(sdf):3d}  Patch={sdf['n_patches'].sum():>8,}")
    for cls in CLASSES:
        cdf = sdf[sdf['label'] == cls]
        if len(cdf) == 0: continue
        print(f"    ↳ {cls:<12} WSI={len(cdf):3d}  Patch={cdf['n_patches'].sum():>8,}  "
              f"[min={cdf['n_patches'].min()}, max={cdf['n_patches'].max()}, "
              f"avg={cdf['n_patches'].mean():.0f}]")
print("═" * 65)

In [ ]:
# ============================================================
# VISUALISASI DISTRIBUSI PATCH
# ============================================================
fig2, axes = plt.subplots(2, 3, figsize=(20, 12), facecolor=BG)
fig2.suptitle('🖼️ Patches_Macenko: Distribusi & Statistik',
              fontsize=18, fontweight='bold', color='white', y=0.99)

# A: Histogram
ax = axes[0,0]; ax.set_facecolor(PANEL)
for cls in CLASSES:
    vals = wsi_df[wsi_df['label']==cls]['n_patches'].values
    ax.hist(vals, bins=20, color=PALETTE[cls], alpha=0.7, label=cls, edgecolor='#30363D')
ax.legend(facecolor='#21262D', edgecolor='#30363D', labelcolor='white')
style_ax(ax, '📈 Histogram Patch per WSI')
ax.set_xlabel('Jumlah Patch', color='#8B949E')
ax.set_ylabel('Frekuensi', color='#8B949E')

# B: Stacked bar WSI per split
ax = axes[0,1]; ax.set_facecolor(PANEL)
slc = wsi_df.groupby(['split','label'])['wsi_id'].count().unstack(fill_value=0)
slc = slc.reindex(['train','validation','test'], fill_value=0)
bot = np.zeros(len(slc))
for cls in CLASSES:
    if cls not in slc.columns: continue
    v = slc[cls].values
    bars = ax.bar(slc.index, v, bottom=bot, color=PALETTE[cls], label=cls,
                  edgecolor='#30363D', linewidth=0.5)
    for bar, val in zip(bars, v):
        if val > 0:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_y()+bar.get_height()/2,
                    str(val), ha='center', va='center', color='white', fontweight='bold')
    bot += v
ax.legend(facecolor='#21262D', edgecolor='#30363D', labelcolor='white')
style_ax(ax, '📊 WSI per Split (Stacked)')
ax.set_ylabel('Jumlah WSI', color='#8B949E')

# C: Violin
ax = axes[0,2]; ax.set_facecolor(PANEL)
data_v = [wsi_df[wsi_df['label']==cls]['n_patches'].values for cls in CLASSES]
data_v = [v if len(v) else np.array([0]) for v in data_v]   # violinplot error jika kosong
parts = ax.violinplot(data_v, positions=range(len(CLASSES)),
                      showmedians=True, showextrema=True)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(list(PALETTE.values())[i]); pc.set_alpha(0.7)
for k in ['cmedians','cbars','cmins','cmaxes']:
    if k in parts: parts[k].set_edgecolor('white')
ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES)
style_ax(ax, '🎻 Violin: Patch per Kelas')
ax.set_ylabel('Jumlah Patch', color='#8B949E')

# D: Total patch per split stacked
ax = axes[1,0]; ax.set_facecolor(PANEL)
pp = wsi_df.groupby(['split','label'])['n_patches'].sum().unstack(fill_value=0)
pp = pp.reindex(['train','validation','test'], fill_value=0)
bot2 = np.zeros(len(pp))
for cls in CLASSES:
    if cls not in pp.columns: continue
    v = pp[cls].values
    ax.bar(pp.index, v, bottom=bot2, color=PALETTE[cls], label=cls,
           edgecolor='#30363D', linewidth=0.5)
    for i, val in enumerate(v):
        if val > 0:
            ax.text(i, bot2[i]+val/2, f'{val:,}',
                    ha='center', va='center', color='white', fontweight='bold', fontsize=9)
    bot2 += v
ax.legend(facecolor='#21262D', edgecolor='#30363D', labelcolor='white')
style_ax(ax, '🔢 Total Patch per Split (Stacked)')
ax.set_ylabel('Jumlah Patch', color='#8B949E')

# E: Top 10 WSI terbanyak patch
ax = axes[1,1]; ax.set_facecolor(PANEL)
t10 = wsi_df.nlargest(10, 'n_patches')[['wsi_id','n_patches','label']]
ax.barh(range(len(t10)), t10['n_patches'].values,
        color=[PALETTE.get(l,ACCENT) for l in t10['label']],
        edgecolor='#30363D')
ax.set_yticks(range(len(t10))); ax.set_yticklabels([w[:18] for w in t10['wsi_id']], fontsize=8)
style_ax(ax, '🏆 Top 10 WSI (Patch Terbanyak)')
ax.set_xlabel('Jumlah Patch', color='#8B949E')
ax.legend(handles=[mpatches.Patch(color=PALETTE[c], label=c) for c in CLASSES],
          facecolor='#21262D', edgecolor='#30363D', labelcolor='white')

# F: Bottom 10
ax = axes[1,2]; ax.set_facecolor(PANEL)
b10 = wsi_df.nsmallest(10, 'n_patches')[['wsi_id','n_patches','label']]
ax.barh(range(len(b10)), b10['n_patches'].values,
        color=[PALETTE.get(l,ACCENT) for l in b10['label']],
        edgecolor='#30363D')
ax.set_yticks(range(len(b10))); ax.set_yticklabels([w[:18] for w in b10['wsi_id']], fontsize=8)
style_ax(ax, '📉 Bottom 10 WSI (Patch Tersedikit)')
ax.set_xlabel('Jumlah Patch', color='#8B949E')
ax.legend(handles=[mpatches.Patch(color=PALETTE[c], label=c) for c in CLASSES],
          facecolor='#21262D', edgecolor='#30363D', labelcolor='white')

plt.tight_layout()
plt.savefig('/tmp/patch_distribution.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("✅ Distribusi patch disimpan.")

In [ ]:
# ============================================================
# CONTOH PATCH — Grid sample per split × kelas (dari ZIP / folder)
# ============================================================
N_SAMPLES = 8
rng = random.Random(42)

def load_random_patch(rec):
    if rec['src_type'] == 'zip':
        names = list_zip_images(rec['wsi_dir'])
        return read_img_from_zip(rec['wsi_dir'], rng.choice(names))
    files = [q for q in Path(rec['wsi_dir']).rglob('*') if q.suffix.lower() in IMG_EXT]
    return Image.open(rng.choice(files)).convert('RGB')

for split in SPLITS:
    sdf = wsi_df[wsi_df['split'] == split]
    if len(sdf) == 0:
        print(f"ℹ️  Split '{split}' tidak punya data — dilewati.")
        continue

    fig3, axes3 = plt.subplots(len(CLASSES), N_SAMPLES,
                               figsize=(N_SAMPLES*2.5, len(CLASSES)*2.8),
                               facecolor=BG, squeeze=False)
    fig3.suptitle(f'🖼️  Sample Patch — Split: {split.upper()}',
                  fontsize=15, fontweight='bold', color='white', y=1.01)

    for ri, cls in enumerate(CLASSES):
        recs = sdf[sdf['label'] == cls].to_dict('records')
        rng.shuffle(recs)
        for ci in range(N_SAMPLES):
            ax = axes3[ri, ci]; ax.set_facecolor(PANEL); ax.axis('off')
            if ci < len(recs):
                try:
                    ax.imshow(np.array(load_random_patch(recs[ci])))
                    ax.set_title(recs[ci]['wsi_id'][:14], fontsize=6, color='#8B949E', pad=2)
                except Exception:
                    ax.text(0.5, 0.5, 'Error', ha='center', va='center',
                            color='red', transform=ax.transAxes)

    fig3.subplots_adjust(left=0.07, wspace=0.05, hspace=0.25)
    for ri, cls in enumerate(CLASSES):
        pos = axes3[ri, 0].get_position()
        fig3.text(pos.x0 - 0.005, pos.y0 + pos.height/2, cls,
                  color=PALETTE.get(cls, 'white'), fontsize=12, fontweight='bold',
                  ha='right', va='center')
    plt.savefig(f'/tmp/sample_patches_{split}.png', dpi=120,
                bbox_inches='tight', facecolor=BG)
    plt.show()
    print(f"✅ Sample patches [{split}] disimpan.")


---
## ⚙️ Sel 3: Setup Ekstraksi Fitur

In [ ]:
# ============================================================
# IMPORT TORCH & SETUP
# ============================================================
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as tv_models
import timm
import h5py
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"   GPU : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ImageNet normalization
TRANSFORM = transforms.Compose([
    transforms.Resize((PATCH_SIZE, PATCH_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])
print("✅ Transform siap")

In [ ]:
# ============================================================
# DATASET & FUNGSI EKSTRAKSI FITUR
# ============================================================

import shutil
LOCAL_CACHE = '/tmp/_zip_cache'   # salinan ZIP sementara di disk lokal (lebih cepat dari Drive)

class WSIPatchDataset(Dataset):
    """Baca semua patch dari satu ZIP atau satu folder WSI."""
    def __init__(self, src: str, transform=None):
        self.src, self.transform = str(src), transform
        self.is_zip = self.src.lower().endswith('.zip')
        if self.is_zip:
            self.names = sorted(list_zip_images(self.src))
        else:
            self.names = sorted(str(p) for p in Path(self.src).rglob('*')
                                if p.suffix.lower() in IMG_EXT)
        if not self.names:
            raise ValueError(f"Tidak ada patch di: {src}")
        self._zf = None   # dibuka malas per worker (ZipFile tidak aman dibagi antar proses)

    def __len__(self): return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]
        if self.is_zip:
            if self._zf is None:
                self._zf = zipfile.ZipFile(self.src)
            image = Image.open(io.BytesIO(self._zf.read(name))).convert('RGB')
        else:
            image = Image.open(name).convert('RGB')
        if self.transform: image = self.transform(image)
        return image, os.path.basename(name)


def extract_features_wsi(model, wsi_dir, out_h5_path,
                          transform, device,
                          batch_size=64, num_workers=2):
    """
    Ekstrak fitur dari satu WSI (ZIP/folder) dan simpan ke out_h5_path.
    H5: 'features' float32 (N, D), 'filenames' str (N,). Return (N, D).
    """
    local_zip = None
    src = str(wsi_dir)
    if src.lower().endswith('.zip'):
        os.makedirs(LOCAL_CACHE, exist_ok=True)
        local_zip = os.path.join(LOCAL_CACHE, os.path.basename(src))
        shutil.copyfile(src, local_zip)      # salin dulu dari Drive → disk lokal
        src = local_zip
    try:
        ds = WSIPatchDataset(src, transform=transform)
        dl = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers,
                        pin_memory=(device.type == 'cuda'))
        all_feats, all_fnames = [], []
        model.eval()
        with torch.no_grad():
            for imgs, fnames in dl:
                feats = model(imgs.to(device))
                all_feats.append(feats.cpu().numpy())
                all_fnames.extend(list(fnames))
        all_feats = np.concatenate(all_feats, axis=0)

        tmp_path = out_h5_path + '.tmp'
        with h5py.File(tmp_path, 'w') as hf:
            hf.create_dataset('features', data=all_feats, dtype='float32',
                              compression='gzip', compression_opts=4)
            hf.create_dataset('filenames', data=np.array(all_fnames, dtype=object),
                              dtype=h5py.string_dtype(encoding='utf-8'))
            hf.attrs['wsi_dir']   = str(wsi_dir)
            hf.attrs['n_patches'] = all_feats.shape[0]
            hf.attrs['feat_dim']  = all_feats.shape[1]
        os.replace(tmp_path, out_h5_path)    # atomic
        return all_feats.shape
    finally:
        if local_zip and os.path.exists(local_zip):
            os.remove(local_zip)


def run_feature_extraction(model_name, feat_dir, wsi_df,
                           transform, device, model,
                           batch_size=64, num_workers=2):
    """
    Loop semua WSI — skip yang sudah selesai (resume support).
    Log disimpan ke feat_dir/extraction_log.csv setelah setiap WSI.
    """
    log_path = os.path.join(feat_dir, 'extraction_log.csv')

    if os.path.exists(log_path):
        log_df    = pd.read_csv(log_path)
        done_wsis = set(log_df[log_df['status'] == 'done']['wsi_id'])
        print(f"🔁 Resume: {len(done_wsis)} WSI sudah done.")
    else:
        log_df    = pd.DataFrame(columns=['wsi_id','split','label',
                                           'n_patches','feat_dim',
                                           'h5_path','status','error'])
        done_wsis = set()

    pbar = tqdm(wsi_df.to_dict('records'), desc=f"[{model_name}]")
    for rec in pbar:
        wsi_id  = rec['wsi_id']
        h5_path = os.path.join(feat_dir, f"{wsi_id}.h5")
        pbar.set_postfix({'wsi': wsi_id[:22], 'split': rec['split']})

        if wsi_id in done_wsis and os.path.exists(h5_path):
            continue   # ← RESUME: skip yang sudah ada

        try:
            shape = extract_features_wsi(
                model, rec['wsi_dir'], h5_path,
                transform, device, batch_size, num_workers
            )
            row = {'wsi_id': wsi_id, 'split': rec['split'],
                   'label': rec['label'], 'n_patches': shape[0],
                   'feat_dim': shape[1], 'h5_path': h5_path,
                   'status': 'done', 'error': ''}
        except Exception as e:
            row = {'wsi_id': wsi_id, 'split': rec['split'],
                   'label': rec['label'], 'n_patches': 0,
                   'feat_dim': 0, 'h5_path': h5_path,
                   'status': 'error', 'error': str(e)}
            print(f"\n⚠️  {wsi_id}: {e}")

        # Simpan log per WSI (checkpoint)
        log_df = pd.concat([log_df[log_df['wsi_id'] != wsi_id],
                            pd.DataFrame([row])], ignore_index=True)
        log_df.to_csv(log_path, index=False)

    pbar.close()
    n_done = (log_df['status']=='done').sum()
    n_err  = (log_df['status']=='error').sum()
    print(f"\n{'='*55}")
    print(f"  [{model_name}] Selesai")
    print(f"  ✅ Done  : {n_done}   ❌ Error: {n_err}")
    print(f"  📂 Output: {feat_dir}")
    print(f"{'='*55}")
    return log_df


print("✅ Semua fungsi ekstraksi siap.")

---
## 🌲 Sel 4: Ekstraksi Fitur — Swin-Tiny

| Item | Detail |
|------|--------|
| Model | `swin_tiny_patch4_window7_224` |
| Pretrained | ImageNet-1k (via `timm`) |
| Feature dim | **768** |
| Output | `Fitur_Swin_Tiny/{WSI_ID}.h5` |

In [ ]:
# ============================================================
# LOAD SWIN-TINY
# ============================================================
print("📥 Loading Swin-Tiny pretrained (ImageNet)...")
swin_model = timm.create_model(
    'swin_tiny_patch4_window7_224',
    pretrained=True,
    num_classes=0,   # hapus classifier head
).to(DEVICE).eval()

with torch.no_grad():
    _out = swin_model(torch.randn(1, 3, 224, 224).to(DEVICE))
    swin_fdim = _out.shape[1]

print(f"✅ Swin-Tiny OK — feature dim: {swin_fdim}")
print(f"   Output folder: {FEAT_SWIN}")
print(f"   Total WSI    : {len(wsi_df)}")

In [ ]:
# ============================================================
# EKSTRAKSI FITUR SWIN-TINY
# Otomatis resume jika runtime terputus
# ============================================================
swin_log = run_feature_extraction(
    model_name  = 'Swin-Tiny',
    feat_dir    = FEAT_SWIN,
    wsi_df      = wsi_df,
    transform   = TRANSFORM,
    device      = DEVICE,
    model       = swin_model,
    batch_size  = BATCH_SIZE,
    num_workers = NUM_WORKERS,
)
display(swin_log.head(10))

In [ ]:
# ============================================================
# VERIFIKASI H5 — SWIN-TINY
# ============================================================
print("🔍 Verifikasi file H5 Swin-Tiny:\n")
for h5p in swin_log[swin_log['status']=='done']['h5_path'].head(5):
    if not os.path.exists(h5p):
        print(f"  ⚠️  Tidak ada: {h5p}"); continue
    with h5py.File(h5p, 'r') as hf:
        print(f"  ✅ {os.path.basename(h5p)}")
        print(f"     features  : {hf['features'].shape}  dtype={hf['features'].dtype}")
        print(f"     filenames : {hf['filenames'].shape}")
        print(f"     File size : {os.path.getsize(h5p)/1e6:.2f} MB\n")

---
## 🧠 Sel 5: Ekstraksi Fitur — DenseNet201

| Item | Detail |
|------|--------|
| Model | `densenet201` |
| Pretrained | ImageNet-1k (via `torchvision`) |
| Feature dim | **1920** |
| Output | `Fitur_DenseNet201/{WSI_ID}.h5` |

In [ ]:
# ============================================================
# LOAD DENSENET201 — hapus classifier head
# ============================================================
print("📥 Loading DenseNet201 pretrained (ImageNet)...")

class DenseNet201Features(nn.Module):
    """DenseNet201 tanpa FC — Global Avg Pool setelah dense block terakhir."""
    def __init__(self):
        super().__init__()
        base = tv_models.densenet201(weights=tv_models.DenseNet201_Weights.DEFAULT)
        self.features = base.features
        self.pool     = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        x = self.features(x)
        x = torch.relu(x)
        x = self.pool(x)
        return x.view(x.size(0), -1)  # (B, 1920)

dense_model = DenseNet201Features().to(DEVICE).eval()

with torch.no_grad():
    _out = dense_model(torch.randn(1, 3, 224, 224).to(DEVICE))
    dense_fdim = _out.shape[1]

print(f"✅ DenseNet201 OK — feature dim: {dense_fdim}")
print(f"   Output folder: {FEAT_DENSE}")
print(f"   Total WSI    : {len(wsi_df)}")

In [ ]:
# ============================================================
# EKSTRAKSI FITUR DENSENET201
# Otomatis resume jika runtime terputus
# ============================================================
dense_log = run_feature_extraction(
    model_name  = 'DenseNet201',
    feat_dir    = FEAT_DENSE,
    wsi_df      = wsi_df,
    transform   = TRANSFORM,
    device      = DEVICE,
    model       = dense_model,
    batch_size  = BATCH_SIZE,
    num_workers = NUM_WORKERS,
)
display(dense_log.head(10))

In [ ]:
# ============================================================
# VERIFIKASI H5 — DENSENET201
# ============================================================
print("🔍 Verifikasi file H5 DenseNet201:\n")
for h5p in dense_log[dense_log['status']=='done']['h5_path'].head(5):
    if not os.path.exists(h5p):
        print(f"  ⚠️  Tidak ada: {h5p}"); continue
    with h5py.File(h5p, 'r') as hf:
        print(f"  ✅ {os.path.basename(h5p)}")
        print(f"     features  : {hf['features'].shape}  dtype={hf['features'].dtype}")
        print(f"     filenames : {hf['filenames'].shape}")
        print(f"     File size : {os.path.getsize(h5p)/1e6:.2f} MB\n")

---
## 📊 Sel 6: Ringkasan Akhir

In [ ]:
# ============================================================
# RINGKASAN TEKS
# ============================================================
print("═" * 65)
print("    📦 RINGKASAN AKHIR EKSTRAKSI FITUR")
print("═" * 65)
for name, log, feat_dir in [
    ('Swin-Tiny',   swin_log,  FEAT_SWIN),
    ('DenseNet201', dense_log, FEAT_DENSE),
]:
    n_done = (log['status']=='done').sum()
    n_err  = (log['status']=='error').sum()
    h5s    = [f for f in os.listdir(feat_dir) if f.endswith('.h5')]
    mb     = sum(os.path.getsize(os.path.join(feat_dir,f)) for f in h5s) / 1e6
    print(f"\n  [{name}]")
    print(f"    ✅ WSI berhasil  : {n_done}/{len(wsi_df)}")
    print(f"    ❌ WSI error     : {n_err}")
    print(f"    📄 File H5       : {len(h5s)}")
    print(f"    💾 Total ukuran  : {mb:.1f} MB")
    print(f"    📂 Output folder : {feat_dir}")
print("\n" + "═"*65)
print("  ✅ Semua proses selesai!")
print("═"*65)

In [ ]:
# ============================================================
# VISUALISASI RINGKASAN STATUS EKSTRAKSI
# ============================================================
fig4, axes4 = plt.subplots(1, 2, figsize=(16, 6), facecolor=BG)
fig4.suptitle('📦 Ringkasan Hasil Ekstraksi Fitur',
              fontsize=16, fontweight='bold', color='white')

for idx, (name, log) in enumerate([
    ('Swin-Tiny', swin_log), ('DenseNet201', dense_log)
]):
    ax = axes4[idx]; ax.set_facecolor(PANEL)
    sc = log['status'].value_counts()
    color_map = {'done': '#2ECC71', 'error': '#E74C3C', 'skip': '#F39C12'}
    clrs = [color_map.get(s, ACCENT) for s in sc.index]
    wedges, texts, autos = ax.pie(
        sc.values, labels=sc.index, colors=clrs,
        autopct='%1.0f%%', startangle=90,
        textprops={'color':'white','fontsize':11},
        wedgeprops={'edgecolor':BG,'linewidth':2},
    )
    for at in autos: at.set_fontweight('bold')
    ax.set_title(f'🔬 {name}\n(Total WSI: {len(log)})',
                 color='white', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/extraction_summary.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("✅ Ringkasan disimpan ke /tmp/extraction_summary.png")